# 04 Limpieza 05 — Corrección final de observaciones pre-deduplicación

Esta libreta es una continuación de la normalización de columnas. No rehace la normalización completa anterior; sólo pule observaciones puntuales detectadas en la base `integrado_solo_unam_estandarizado_pre_deduplicacion.csv`.

**Entrada:** `C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion.csv`

**Salida única:** `C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv`

No deduplica, no toca `Autor_norm`, no cambia `Afiliacion1`, `Afiliacion2` ni `Area`, y mantiene las 14 columnas canónicas.

In [1]:
from __future__ import annotations

import csv
import html
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

RAIZ_PROYECTO = Path(r"C:\Users\hazar\Desktop\PROYECTO")
CARPETA_NORMALIZAR_COLUMNAS = RAIZ_PROYECTO / "04_Limpieza" / "02_Normalizar_Columnas"
CARPETA_NORMALIZAR_COLUMNAS.mkdir(parents=True, exist_ok=True)

INPUT_CSV = CARPETA_NORMALIZAR_COLUMNAS / "integrado_solo_unam_estandarizado_pre_deduplicacion.csv"
OUTPUT_CSV = CARPETA_NORMALIZAR_COLUMNAS / "integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv"


CANONICAL_COLUMNS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

FALSE_EMPTY = {
    "", "n/d", "nd", "n.a", "n/a", "na", "nan", "none", "null", "sin informacion",
    "sin información", "sin dato", "sin datos", "no disponible", "not available", "-", "--",
}

URL_RE = re.compile(r"(?i)\b(?:https?://|www\.)\S+")
REAL_HTML_TAG_RE = re.compile(
    r"</?\s*(?:sup|sub|br|p|div|span|i|b|em|strong|italic|bold|html|body|table|tr|td|th|ul|ol|li|a)(?:\s+[^>]*)?>",
    flags=re.I,
)
LATEX_TRIGGER_RE = re.compile(
    r"\\|\$|\b(?:mathsf|mathrm|mathbf|mathbb|mathcal|mathop|operatorname|textit|textbf|frac|varepsilon)\b|"
    r"\b(?:Theta|Omega)\s+left\s*\(|\bOleft\s*\(|\bleft\s*\(|\bright\s*\)|\bS5_n\b|\bN_\s*Q\b|\bepsilon\s+-",
    flags=re.I,
)
COPYRIGHT_RE = re.compile(
    r"(?i)\b(copyright|all rights reserved|creative commons|licensed under|license|full-text articles|authorized sources)\b|\[\s*The copyright"
)
AVAILABILITY_TERM_RE = re.compile(
    r"(?i)\b(?:code is available|our code|source code|the source code|project page|"
    r"database url|data are available|dataset is available|available at|freely available|"
    r"available for use|implementation is available|software is available|github|zenodo|"
    r"repository|git repository|source-code|codes are available|tool is available)\b"
)
BULLET_RE = re.compile(r"[•●▪◦]")
DISPLAY_RE = re.compile(r"(?i)\[\s*Display omitted\s*\]")
MOJIBAKE_RE = re.compile(r"(?:Ã.|Â|â€|�|ĝ)")

# ---------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------
def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(text or "")) if not unicodedata.combining(ch))


def norm_key(text: str) -> str:
    text = strip_accents(str(text or "")).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_spaces(text: str) -> str:
    text = html.unescape("" if text is None else str(text))
    text = text.replace("\ufeff", " ").replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\b(E_γ)\s*-\s*", r"\1-", text)
    text = re.sub(r"\b(ϵ)\s*-\s*", r"\1-", text)
    text = re.sub(r"\b([a-záéíóúñ]{2,})-\s+([a-záéíóúñ]{2,})\b", r"\1\2", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def blank_false_empty(text: str) -> str:
    text = clean_spaces(text)
    return "" if norm_key(text) in FALSE_EMPTY else text


def join_unique(values: Iterable[str]) -> str:
    out, seen = [], set()
    for value in values:
        value = clean_spaces(value)
        if not value:
            continue
        key = norm_key(value)
        if key not in seen:
            seen.add(key)
            out.append(value)
    return ", ".join(out)


def strip_real_html(text: str) -> str:
    text = html.unescape(str(text or ""))
    text = REAL_HTML_TAG_RE.sub(" ", text)
    return clean_spaces(text)

# ---------------------------------------------------------------------
# Índice, Año, DOI, URL, ISBN, ISSN
# ---------------------------------------------------------------------
def clean_indice(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    m = re.search(r"\d+(?:\.0+)?", text)
    return str(int(float(m.group(0)))) if m else ""


def clean_year(text: str) -> str:
    text = blank_false_empty(text)
    m = re.search(r"\b(2024|2025)\b", text or "")
    return m.group(1) if m else ""


def clean_doi(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    text = strip_real_html(text)
    text = re.sub(r"(?i)^\s*(?:https?://(?:dx\.)?doi\.org/|(?:dx\.)?doi\.org/|doi\s*[:/]\s*|doi/)\s*", "", text)
    m = re.search(r"(?i)10\.\d{4,9}/[^\s,;<>\"]+", text)
    if not m:
        return ""
    return m.group(0).strip().strip(" .;,)]}>").lower()


def clean_url(text: str, doi: str) -> str:
    text = blank_false_empty(text)
    doi = clean_doi(doi)
    if text:
        text = re.sub(r"\s+", "", text)
        if clean_doi(text) and doi:
            return f"https://doi.org/{doi}"
        if text.lower().startswith("doi.org/"):
            return "https://" + text
        if text.lower().startswith("dx.doi.org/"):
            return "https://doi.org/" + text.split("/", 1)[1]
        if text.lower().startswith("http://doi.org/") or text.lower().startswith("http://dx.doi.org/"):
            return re.sub(r"(?i)^http://(?:dx\.)?doi\.org/", "https://doi.org/", text)
        if text.lower().startswith("https://dx.doi.org/"):
            return re.sub(r"(?i)^https://dx\.doi\.org/", "https://doi.org/", text)
        if text.lower().startswith(("http://", "https://")):
            return text
    return f"https://doi.org/{doi}" if doi else ""


def clean_isbn(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    if re.search(r"(?i)\d+(?:\.\d+)?\s*e\s*\+\s*\d+", text):
        return ""  # no se adivina ISBN en notación científica
    text = strip_real_html(text)
    text = re.sub(r"(?i)\bISBN(?:-1[03])?\s*[:=]?\s*", "", text)
    text = re.sub(r"[;|]", ",", text)
    candidates = []
    for part in text.split(","):
        m = re.search(r"(?i)(?:97[89][- ]?)?\d[-\d ]{7,}[\dX]", part)
        if m:
            isbn = re.sub(r"\s+", "-", m.group(0).upper().strip(" -"))
            isbn = re.sub(r"-+", "-", isbn)
            candidates.append(isbn)
    return join_unique(candidates)


def clean_issn(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    text = strip_real_html(text)
    text = re.sub(r"(?i)\b(?:e?issn|print issn|online issn|issn-l)\s*[:=]?\s*", "", text)
    candidates = re.findall(r"(?i)\b\d{4}[- ]?\d{3}[0-9X]\b", text)
    out = []
    for cand in candidates:
        digits = re.sub(r"[^0-9Xx]", "", cand).upper()
        if len(digits) == 8:
            out.append(digits[:4] + "-" + digits[4:])
    return join_unique(out)

# ---------------------------------------------------------------------
# LaTeX/residuos matemáticos
# ---------------------------------------------------------------------
def convert_latex_minimal(text: str) -> str:
    """Sólo se llama cuando hay señales de LaTeX/residuos; no toca textos normales."""
    text = str(text or "")
    text = html.unescape(text)

    # Comandos con llaves primero.
    for cmd in ["mathrm", "mathsf", "mathbf", "mathbb", "mathcal", "mathit", "textit", "textbf", "operatorname"]:
        text = re.sub(rf"\\{cmd}\s*\{{([^{{}}]*)\}}", r"\1", text)
    text = re.sub(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"\1/\2", text)
    text = re.sub(r"_\s*\{([^{}]+)\}", r"_\1", text)
    text = re.sub(r"\^\s*\{([^{}]+)\}", r"^\1", text)

    # Comandos griegos.
    greek = {
        r"\varepsilon": "ϵ", r"\epsilon": "ϵ", r"\gamma": "γ", r"\Gamma": "Γ",
        r"\theta": "θ", r"\Theta": "Θ", r"\omega": "ω", r"\Omega": "Ω", r"\Delta": "Δ",
    }
    for old, new in greek.items():
        text = text.replace(old, new)

    # Residuos sin barra por conversión previa.
    replacements = [
        (r"\bmathsf\s+E\s*_\s*gamma\b", "E_γ"),
        (r"\bmathsf\s+E\s*_\s*γ\b", "E_γ"),
        (r"\bE\s*_\s*gamma\b", "E_γ"),
        (r"\bI_\s*mathrm\s+EDR\b", "I_EDR"),
        (r"\bmathrm\s+EDR\b", "EDR"),
        (r"\bmathbf\s+S5_n\b", "S5_n"),
        (r"\bS5n\s+S5_n\b", "S5_n"),
        (r"\bTheta\s+left\s*\(", "Θ("),
        (r"\bOmega\s+left\s*\(", "Ω("),
        (r"\bOleft\s*\(", "O("),
        (r"\bleft\s*\(", "("),
        (r"\bright\s*\)", ")"),
        (r"\bright\s*\.", ")"),
        (r"\bfrac\s+1\s+varepsilon\b", "1/ϵ"),
        (r"\bfrac\s+1\s+ϵ\b", "1/ϵ"),
        (r"\blog\s+frac\s+1\s+varepsilon\b", "log 1/ϵ"),
        (r"\blog\s+frac\s+1\s+ϵ\b", "log 1/ϵ"),
        (r"\bvarepsilon\b", "ϵ"),
        (r"\bepsilon\s+-\s*locally\b", "ϵ-locally"),
        (r"\bepsilon\s*\(\+\)", "ϵ(+)"),
        (r"\bN_\s*Q\s*,\s*epsilon\b", "N_Q,ϵ"),
        (r"\bN-Q\s*,\s*N-epsilon\b", "N_Q,ϵ"),
        (r"\bL_\s*Q\s*,\s*epsilon\b", "L_Q,ϵ"),
        (r"\bArchiveUpdateL_\s*Q\s*,\s*epsilon\b", "ArchiveUpdateL_Q,ϵ"),
        (r"\bS5_n\b", "S5n"),
        (r"\bN_Q\s*,\s*ϵ\b", "NQ,ϵ"),
        (r"\bN_Q\b", "NQ"),
        (r"\bL_Q\s*,\s*ϵ\b", "LQ,ϵ"),
        (r"\bL_Q\b", "LQ"),
        (r"\bArchiveUpdateL_Q\s*,\s*ϵ\b", "ArchiveUpdateLQ,ϵ"),
        (r"\bepsilon\s+-\s*acceptable\b", "ϵ-acceptable"),
    ]
    for pat, repl in replacements:
        text = re.sub(pat, repl, text, flags=re.I)

    # Quitar comandos/remanentes explícitos sin borrar palabras normales left/right.
    text = re.sub(r"\\[A-Za-z]+", " ", text)
    text = text.replace("$", " ").replace("{", " ").replace("}", " ").replace("\\", " ")
    text = re.sub(r"\b(?:mathsf|mathrm|mathbf|mathbb|mathcal|mathop|operatorname|textit|textbf)\b", " ", text, flags=re.I)
    text = re.sub(r"\s*_\s*", "_", text)
    text = re.sub(r"\s*\^\s*", "^", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ---------------------------------------------------------------------
# Título
# ---------------------------------------------------------------------
def clean_title(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    text = strip_real_html(text)
    if LATEX_TRIGGER_RE.search(text):
        text = convert_latex_minimal(text)
    text = re.sub(r"\bNeuralode\b", "NeuralODE", text, flags=re.I)
    # Correcciones puntuales de artefactos de extracción en títulos.
    text = re.sub(r"\bUncertainty\s*!antification\b", "Uncertainty Quantification", text, flags=re.I)
    text = re.sub(r"\bf\s+-\s*Divergences\b", "f-Divergences", text)
    text = re.sub(r":\s*:\s*", ": ", text)
    text = re.sub(r"\s+:\s+", ": ", text)
    if text and text.upper() == text and re.search(r"[A-ZÁÉÍÓÚÑ]", text):
        text = text.title()
        text = re.sub(r"\bNeuralode\b", "NeuralODE", text, flags=re.I)
        text = re.sub(r"\bOde\b", "ODE", text)
        text = re.sub(r"\bPde\b", "PDE", text)
        text = re.sub(r"\bPet\b", "PET", text)
        text = re.sub(r"\bMri\b", "MRI", text)
        text = re.sub(r"\bAi\b", "AI", text)
    return clean_spaces(text)

# ---------------------------------------------------------------------
# Keywords
# ---------------------------------------------------------------------
def keyword_dedupe_key(keyword: str) -> str:
    key = strip_accents(keyword).lower()
    key = re.sub(r"<[^>]+>", " ", key)
    key = re.sub(r"https?://\S+|www\.\S+", " ", key)
    key = re.sub(r"[^a-z0-9]+", " ", key)
    return re.sub(r"\s+", " ", key).strip()


def clean_keywords(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    text = html.unescape(text)
    text = REAL_HTML_TAG_RE.sub(" ", text)
    text = re.sub(r"(?i)\b(?:keywords?|author keywords?|index terms?|ccs concepts?)\s*[:：]", " ", text)
    text = URL_RE.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", ",")
    text = re.sub(r"\s*[;|]\s*", ",", text)
    text = re.sub(r"\s*,\s*", ",", text)
    parts = [p.strip(" .,:;|\t\n\r") for p in text.split(",") if p.strip(" .,:;|\t\n\r")]
    # Basura editorial detectada en algunas fuentes/plantillas IEEE.
    editorial_keys = {"component", "formatting", "style", "styling", "insert"}
    part_keys_preview = {keyword_dedupe_key(p) for p in parts if keyword_dedupe_key(p)}
    if part_keys_preview and part_keys_preview.issubset(editorial_keys):
        return ""
    seen = set()
    out = []
    for part in parts:
        # Regla del proyecto: keywords en minúsculas; acrónimos se conservan sólo si ya son token entero en mayúsculas.
        tokens = []
        for tok in part.split():
            if re.fullmatch(r"[A-Z]{2,}\d*", tok):
                tokens.append(tok)
            else:
                tokens.append(tok.lower())
        part_clean = clean_spaces(" ".join(tokens))
        key = keyword_dedupe_key(part_clean)
        if key and key not in seen:
            seen.add(key)
            out.append(part_clean)
    return ", ".join(out)

# ---------------------------------------------------------------------
# Abstract
# ---------------------------------------------------------------------
def replace_mojibake(text: str) -> str:
    replacements = {
        "ĝ.,•": "ℕ", "ĝ., •": "ℕ", "Â": "", "â€“": "–", "â€”": "—",
        "â€˜": "‘", "â€™": "’", "â€œ": "“", "â€\u009d": "”",
        "Ã¡": "á", "Ã©": "é", "Ã­": "í", "Ã³": "ó", "Ãº": "ú", "Ã±": "ñ",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text


def remove_display_and_bullets(text: str) -> str:
    if DISPLAY_RE.search(text):
        text = re.sub(r"\s*\[\s*Display omitted\s*\].*$", "", text, flags=re.I | re.S)
    # Cortar highlights añadidos al final. Sólo si hay texto suficiente antes del bullet.
    m = re.search(r"\s+[•●▪◦]\s+(?=[A-ZÁÉÍÓÚ0-9])", text)
    if m and m.start() > 80:
        text = text[:m.start()]
    # Si el abstract empieza con bullets, eliminarlos.
    while re.match(r"^\s*[•●▪◦]\s+", text):
        text = re.sub(r"^\s*[•●▪◦]\s+.*?(?:\.\s+|$)", "", text, count=1, flags=re.S)
    return clean_spaces(text)


def remove_copyright_tail(text: str) -> str:
    patterns = [
        r"\s*\[[^\]]*(?:copyright|full-text articles|authorized sources|publisher|rights reserved|license)[^\]]*\]\s*$",
        r"\s*(?:©|\(c\))\s*\d{4}.*$",
        r"\s*Copyright\s+Author\s*\(?s\)?\s*\d{4}\.?\s*$",
        r"\s*Copyright\s+Author\s*\(\s*s\s*\)\s*\d{4}\.?\s*$",
        r"\s*Copyright\s*\d{4}\.?\s*$",
        r"\s*Copyright\.?\s*$",
        r"\s*All rights reserved\.?\s*$",
        r"\s*This is an open access article.*$",
        r"\s*Licensed under.*$",
        r"\s*Creative Commons.*$",
        r"\s*Availability and implementation\b.*$",
        r"\s*Database URL\s*[:：].*$",
    ]
    old = None
    while old != text:
        old = text
        for pat in patterns:
            text = re.sub(pat, "", text, flags=re.I | re.S)
        text = clean_spaces(text)
    if re.fullmatch(r"(?is)(copyright|©|all rights reserved|license|licensed|creative commons).{0,150}", text.strip()):
        return ""
    return text


def remove_availability_sentences_and_urls(text: str) -> str:
    if not (URL_RE.search(text) or AVAILABILITY_TERM_RE.search(text)):
        return text

    sentence_patterns = [
        r"(?:^|(?<=[.!?])\s+)(?:Our\s+code|Code|Source\s+code|The\s+source\s+code|Our\s+source\s+code|Implementation|Project\s+page|The\s+project\s+page|Our\s+code\s+and\s+trained\s+models|Our\s+code\s+and\s+synthetic\s+datasets|Data|Dataset|The\s+dataset)[^.?!]{0,450}(?:[.!?]|$)",
        r"(?:^|(?<=[.!?])\s+)(?:For\s+seamless\s+access|The\s+database\s+is\s+available|The\s+tool\s+is\s+available|BIOMX-DB\s+is\s+freely\s+available)[^.?!]{0,450}(?:[.!?]|$)",
        r"(?:^|(?<=[.!?])\s+)Availability and implementation[^.?!]{0,450}(?:[.!?]|$)",
        r"(?:^|(?<=[.!?])\s+)Database URL\s*[:：][^.?!]{0,450}(?:[.!?]|$)",
        r"(?:^|(?<=[.!?])\s+)[^.?!]{0,260}\b(?:source code|git repository|repository|freely available|available for use|available at|github|zenodo)\b[^.?!]{0,260}(?:[.!?]|$)",
    ]
    old = None
    while old != text:
        old = text
        for pat in sentence_patterns:
            text = re.sub(pat, " ", text, flags=re.I | re.S)
        text = clean_spaces(text)

    # URLs bibliográficas restantes se eliminan, no la frase completa.
    text = URL_RE.sub("", text)
    # Si quedó sólo la etiqueta, eliminarla.
    text = re.sub(r"(?i)\bDatabase URL\s*[:：]?\s*$", "", text)
    text = re.sub(r"\(\s*\)", "", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r";\s*;", ";", text)
    return clean_spaces(text)


def clean_abstract(text: str) -> str:
    text = blank_false_empty(text)
    if not text:
        return ""
    text = html.unescape(text)
    text = REAL_HTML_TAG_RE.sub(" ", text)
    text = re.sub(r"(?i)^\s*abstract\s*[:.\-–—]?\s*", "", text)
    text = replace_mojibake(text)
    if DISPLAY_RE.search(text) or BULLET_RE.search(text):
        text = remove_display_and_bullets(text)
    if URL_RE.search(text) or AVAILABILITY_TERM_RE.search(text):
        text = remove_availability_sentences_and_urls(text)
    if COPYRIGHT_RE.search(text):
        text = remove_copyright_tail(text)
    if LATEX_TRIGGER_RE.search(text):
        text = convert_latex_minimal(text)
    if COPYRIGHT_RE.search(text):
        text = remove_copyright_tail(text)
    text = clean_spaces(text)
    # Reparar oración pegada tras extracción de fuente.
    text = re.sub(r"(?<=[.!?])(?=[A-ZÁÉÍÓÚÑ])", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\b(E_γ)\s*-\s*", r"\1-", text)
    text = re.sub(r"\b(ϵ)\s*-\s*", r"\1-", text)
    text = re.sub(r"\b([a-záéíóúñ]{2,})-\s+([a-záéíóúñ]{2,})\b", r"\1\2", text)
    text = re.sub(r"\(\s*\)", "", text)
    text = clean_spaces(text).strip(" ;,")
    if not text or re.fullmatch(r"(?is)(copyright|license|licensed|all rights reserved|creative commons|open access).{0,150}", text):
        return ""
    return text

# ---------------------------------------------------------------------
# Procesamiento y validación
# ---------------------------------------------------------------------
def process_row(row: Dict[str, str], row_number: int) -> Tuple[Dict[str, str], List[Dict[str, str]]]:
    original = {col: row.get(col, "") or "" for col in CANONICAL_COLUMNS}
    out = {col: blank_false_empty(original[col]) for col in CANONICAL_COLUMNS}

    out["indice"] = clean_indice(out["indice"])
    out["Titulo"] = clean_title(out["Titulo"])
    out["Año"] = clean_year(out["Año"])
    out["ISBN"] = clean_isbn(out["ISBN"])
    out["ISSN"] = clean_issn(out["ISSN"])
    out["Doi"] = clean_doi(out["Doi"])
    out["URL"] = clean_url(out["URL"], out["Doi"])
    out["Subarea"] = ""
    out["Keywords"] = clean_keywords(out["Keywords"])
    out["Abstract"] = clean_abstract(out["Abstract"])

    # No normalizar estas columnas en esta fase.
    out["Autor_norm"] = blank_false_empty(original["Autor_norm"])
    out["Afiliacion1"] = blank_false_empty(original["Afiliacion1"])
    out["Afiliacion2"] = blank_false_empty(original["Afiliacion2"])
    out["Area"] = blank_false_empty(original["Area"])

    changes = []
    for col in CANONICAL_COLUMNS:
        before, after = original[col], out[col]
        if before != after:
            changes.append({
                "csv_row": str(row_number),
                "indice": out.get("indice", ""),
                "Doi": out.get("Doi", ""),
                "Titulo": out.get("Titulo", ""),
                "Autor_norm": out.get("Autor_norm", ""),
                "columna": col,
                "valor_antes": before,
                "valor_despues": after,
            })
    return out, changes


def keyword_has_soft_duplicates(text: str) -> bool:
    keys = [keyword_dedupe_key(p) for p in str(text or "").split(",") if keyword_dedupe_key(p)]
    return len(keys) != len(set(keys))


def has_display_bullet_residual(text: str) -> bool:
    text = str(text or "")
    return bool(DISPLAY_RE.search(text) or re.search(r"^\s*[•●▪◦]\s+|\s[•●▪◦]\s+(?=[A-ZÁÉÍÓÚ0-9])", text))


def validate_rows(rows: List[Dict[str, str]]) -> Dict[str, int]:
    counts = Counter()
    for r in rows:
        if list(r.keys()) != CANONICAL_COLUMNS:
            counts["columnas_no_canonicas"] += 1
        if not re.fullmatch(r"\d+", r.get("indice", "")):
            counts["indice_no_entero"] += 1
        if not r.get("Titulo", "").strip():
            counts["titulo_vacio"] += 1
        if "Neuralode" in r.get("Titulo", ""):
            counts["titulo_neuralode"] += 1
        if LATEX_TRIGGER_RE.search(r.get("Titulo", "")):
            counts["titulo_latex_residual"] += 1
        if r.get("Año", "") not in {"2024", "2025"}:
            counts["anio_fuera_2024_2025"] += 1
        if "E+" in r.get("ISBN", "").upper():
            counts["isbn_notacion_cientifica"] += 1
        issn = r.get("ISSN", "")
        if issn:
            for part in [p.strip() for p in issn.split(",") if p.strip()]:
                if not re.fullmatch(r"\d{4}-\d{3}[0-9X]", part):
                    counts["issn_formato_invalido"] += 1
                    break
        doi = r.get("Doi", "")
        if doi and not doi.startswith("10."):
            counts["doi_no_inicia_10"] += 1
        url = r.get("URL", "")
        if url and not url.lower().startswith("http"):
            counts["url_no_http"] += 1
        if r.get("Subarea", ""):
            counts["subarea_no_vacia"] += 1
        kw = r.get("Keywords", "")
        if REAL_HTML_TAG_RE.search(kw):
            counts["keywords_html"] += 1
        if URL_RE.search(kw):
            counts["keywords_url"] += 1
        if ";" in kw or "|" in kw:
            counts["keywords_separador_invalido"] += 1
        if keyword_has_soft_duplicates(kw):
            counts["keywords_duplicadas_suaves"] += 1
        abstract = r.get("Abstract", "")
        if REAL_HTML_TAG_RE.search(abstract):
            counts["abstract_html"] += 1
        if URL_RE.search(abstract):
            counts["abstract_url"] += 1
        if AVAILABILITY_TERM_RE.search(abstract):
            counts["abstract_disponibilidad"] += 1
        if COPYRIGHT_RE.search(abstract):
            counts["abstract_copyright"] += 1
        if has_display_bullet_residual(abstract):
            counts["abstract_display_bullets"] += 1
        if LATEX_TRIGGER_RE.search(abstract):
            counts["abstract_latex"] += 1
        if MOJIBAKE_RE.search(abstract):
            counts["abstract_mojibake"] += 1
        for col in CANONICAL_COLUMNS:
            val = r.get(col, "")
            if val.strip() and norm_key(val) in FALSE_EMPTY:
                counts[f"vacio_falso_{col}"] += 1
    return dict(counts)


def read_csv_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != CANONICAL_COLUMNS:
            raise ValueError(f"Columnas inesperadas. Esperadas={CANONICAL_COLUMNS}; recibidas={reader.fieldnames}")
        return [{col: row.get(col, "") or "" for col in CANONICAL_COLUMNS} for row in reader]


def write_csv_rows(path: Path, rows: List[Dict[str, str]], columns: List[str]) -> None:
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)




In [2]:
def main() -> None:
    rows = read_csv_rows(INPUT_CSV)
    cleaned_rows: List[Dict[str, str]] = []
    total_changes: List[Dict[str, str]] = []

    for row_number, row in enumerate(rows, start=2):
        cleaned, changes = process_row(row, row_number)
        cleaned_rows.append(cleaned)
        total_changes.extend(changes)

    write_csv_rows(OUTPUT_CSV, cleaned_rows, CANONICAL_COLUMNS)

    changed_by_col = Counter(d["columna"] for d in total_changes)
    validation = validate_rows(cleaned_rows)
    summary = {
        "input_csv": str(INPUT_CSV),
        "output_csv": str(OUTPUT_CSV),
        "filas_entrada": len(rows),
        "filas_salida": len(cleaned_rows),
        "columnas_finales": CANONICAL_COLUMNS,
        "filas_con_alguna_modificacion": len({d["csv_row"] for d in total_changes}),
        "celdas_modificadas_por_columna": dict(changed_by_col),
        "validacion_final_observaciones": validation,
        "validacion_total_observaciones": int(sum(validation.values())),
    }

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("\nArchivo final generado:")
    print(OUTPUT_CSV)


if __name__ == "__main__":
    main()


{
  "input_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\02_Normalizar_Columnas\\integrado_solo_unam_estandarizado_pre_deduplicacion.csv",
  "output_csv": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\02_Normalizar_Columnas\\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv",
  "filas_entrada": 3239,
  "filas_salida": 3239,
  "columnas_finales": [
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "Subarea",
    "Keywords",
    "Abstract"
  ],
  "filas_con_alguna_modificacion": 422,
  "celdas_modificadas_por_columna": {
    "Abstract": 357,
    "Keywords": 65,
    "Titulo": 12
  },
  "validacion_final_observaciones": {},
  "validacion_total_observaciones": 0
}

Archivo final generado:
C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv
